In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_product_qc(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "ProductId  string "
        #     ",SourceSystem string"
        #     ",IngestionTimestamp timestamp"
        #     ",BatchId string"
        #     ",ProductKey  string "
        #     ",ProductCode  string "
        #     ",ProductDescription  string"
        #     ",SubProductCode  int"
        #     ",SubProduct  string"
        #     ",BankingUnitCode  string"
        #     ",InterestUnit  string"
        #     ",InterestFrequency  int"
        #     ",DeliquentInterestUnit  string"
        #     ",DeliquentInterestFrequency  int"
        #     ",InterestDate  date"
        #     ",LastValeurDate  date"
        #     ",NextValeurDate  date"
        #     ",PeriodicIntFlag  string"
        #     ",ShadowFlag  boolean"
        #     ",SyndicatedFlag  boolean"
        #     ",L3869Flag  boolean"
        #     ",OCWFlag  boolean"
        #     ",BondedFlag  boolean"
        #     ",ProductCategory  string"
        #     ",BusinessDate  date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Product"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qc",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ========================== subproduct ==========================
        subproduct_extra_filters = None

        if subproduct_extra_filters is not None:
            subproduct_extra_filters.append(date_filter)
            subproduct_final_filters = " AND ".join(subproduct_extra_filters)
        else:
            subproduct_final_filters = date_filter

        subproduct_selected_fields = [
            "sprd_code",
            "sprd_desc",
            "sprd_intcode",
            "businessdate",
            "prd_desc",
        ]

        subproduct = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_subproduct",
            subproduct_final_filters,
            subproduct_selected_fields,
        )

        subproduct_primary_key_fields = [subproduct.sprd_code, lit("qc")]

        product_qc = (
            subproduct.withColumn(
                "ProductId", sha2(concat_ws(";", *subproduct_primary_key_fields), 256)
            )
            .withColumn("ProductCode", subproduct.sprd_code.cast("string"))
            .withColumn("ProductKey", subproduct.sprd_code.cast("string"))
            .withColumn("ProductDescription", subproduct.sprd_desc.cast("string"))
            .withColumn("BankingUnitCode", subproduct.sprd_intcode.cast("string"))
            .withColumn("InterestUnit", lit(None).cast("string"))
            .withColumn("InterestFrequency", lit(None).cast("int"))
            .withColumn("DeliquentInterestUnit", lit(None).cast("string"))
            .withColumn("DeliquentInterestFrequency", lit(None).cast("int"))
            .withColumn("InterestDate", lit(None).cast("date"))
            .withColumn("LastValeurDate", lit(None).cast("date"))
            .withColumn("NextValeurDate", lit(None).cast("date"))
            .withColumn("PeriodicIntFlag", lit(None).cast("string"))
            .withColumn("ProductCategory",subproduct.prd_desc.cast("string"))
        )

        product_qc = product_qc.select(
            product_qc.ProductId,
            lit("qc").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            product_qc.ProductKey,
            product_qc.ProductCode,
            product_qc.ProductDescription,
            lit(None).cast("int").alias("SubProductCode"),
            lit(None).cast("string").alias("SubProduct"),
            product_qc.BankingUnitCode,
            product_qc.InterestUnit,
            product_qc.InterestFrequency,
            product_qc.DeliquentInterestUnit,
            product_qc.DeliquentInterestFrequency,
            product_qc.InterestDate,
            product_qc.LastValeurDate,
            product_qc.NextValeurDate,
            product_qc.PeriodicIntFlag,
            lit(None).cast("boolean").alias("ShadowFlag"),
            lit(None).cast("boolean").alias("SyndicatedFlag"),
            lit(None).cast("boolean").alias("L3869Flag"),
            lit(None).cast("boolean").alias("OCWFlag"),
            lit(None).cast("boolean").alias("BondedFlag"),
            product_qc.ProductCategory,
            product_qc.businessdate.alias("BusinessDate"),
        )

        # ================== Append ==================

        product_qc.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_product_qc(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)